In [12]:
# -----------------------------
# 0. IMPORTS
# -----------------------------
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# download VADER lexicon
nltk.download("vader_lexicon")

# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv(
    r"C:\Users\ZAK-TECH\Desktop\KAIM week 2 new\solar-challenge-week2\Scraper\data\processed\reviews_processed.csv"
)

print("Loaded dataset:")
print(df.head())

print("\nReview count per bank:")
print(df["bank_name"].value_counts())

# -----------------------------
# 2. SENTIMENT ANALYSIS (NLTK)
# -----------------------------
sia = SentimentIntensityAnalyzer()

def compute_sentiment(text):
    if pd.isna(text):
        return 0
    return sia.polarity_scores(str(text))["compound"]

df["sentiment_score"] = df["review_text"].apply(compute_sentiment)

def label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment_score"].apply(label)

# Save intermediate
df.to_csv("task2_with_sentiment.csv", index=False)
print("\nSaved: task2_with_sentiment.csv")

# -----------------------------
# 3. AGGREGATION: sentiment per bank/rating
# -----------------------------
agg = df.groupby(["bank_name", "rating"])["sentiment_score"].agg(["count", "mean"])
print("\nSentiment summary per bank/rating:")
print(agg.sort_values("count", ascending=False).head(20))

# -----------------------------
# 4. THEMATIC ANALYSIS — TF-IDF keywords per bank
# -----------------------------
def extract_keywords_per_bank(df, bank_col="bank_name", text_col="review_text", top_k=30):
    bank_keywords = {}

    for bank in df[bank_col].unique():
        texts = df[df[bank_col] == bank][text_col].dropna().astype(str)

        if len(texts) == 0:
            continue

        vectorizer = TfidfVectorizer(
            max_features=3000,
            ngram_range=(1, 2),
            min_df=2,
            stop_words="english"
        )
        tfidf = vectorizer.fit_transform(texts)
        scores = tfidf.sum(axis=0).A1
        words = vectorizer.get_feature_names_out()

        sorted_idx = scores.argsort()[::-1]
        keywords = [(words[i], scores[i]) for i in sorted_idx[:top_k]]

        bank_keywords[bank] = keywords

    return bank_keywords

bank_keywords = extract_keywords_per_bank(df)

# print sample
for bank, kws in bank_keywords.items():
    print(f"\nTop keywords for {bank}:")
    for w, s in kws[:15]:
        print(f"  {w} ({s:.2f})")

# -----------------------------
# 5. THEME MAPPING (RULE BASED)
# -----------------------------
theme_mapping = {
    "Account Access Issues": ["login", "password", "pin", "fingerprint", "authenticate", "access"],
    "Transaction Performance": ["slow", "delay", "transfer", "processing", "timeout", "speed"],
    "UI & UX": ["ui", "interface", "design", "layout", "buttons", "navigation"],
    "Crashes & Stability": ["crash", "freeze", "bug", "error", "stuck", "hang"],
    "Customer Support": ["support", "help", "customer service", "agent", "call"]
}

# Cluster fallback
def cluster_keywords(words, n_clusters=3):
    if len(words) < n_clusters:
        return {"cluster_0": words}

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(words)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(X)

    clusters = {}
    for word, label in zip(words, labels):
        clusters.setdefault(f"cluster_{label}", []).append(word)

    return clusters

# assign themes per bank
bank_themes = {}

for bank, kws in bank_keywords.items():
    keywords_only = [k for k, _ in kws]

    matched = []
    for theme, terms in theme_mapping.items():
        for kw in keywords_only:
            if any(t in kw for t in terms):
                matched.append(theme)
                break

    # fallback
    if len(matched) < 2:
        clusters = cluster_keywords(keywords_only[:30], n_clusters=3)
        matched = list(clusters.keys())

    bank_themes[bank] = matched

print("\nSuggested themes per bank:")
for bank, t in bank_themes.items():
    print(bank, "=>", t)

# -----------------------------
# 6. ASSIGN THEMES TO EACH REVIEW
# -----------------------------
def detect_themes(text, mapping):
    text = str(text).lower()
    found = []

    for theme, words in mapping.items():
        for w in words:
            if w in text:
                found.append(theme)
                break

    return ";".join(sorted(found)) if found else "Other"

df["identified_themes"] = df["review_text"].apply(lambda t: detect_themes(t, theme_mapping))

# -----------------------------
# 7. SAVE FINAL RESULTS
# -----------------------------
out_cols = [
    "review_text", "rating", "date", "bank_name", "source",
    "sentiment_score", "sentiment_label", "identified_themes"
]

df[out_cols].to_csv("task2_results.csv", index=False)
print("\nSaved final output: task2_results.csv")


[nltk_data] Downloading package vader_lexicon to C:\Users\ZAK-
[nltk_data]     TECH\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Loaded dataset:
                              review_id review_text  rating review_date  \
0  3463230e-f9f7-4be3-a632-fdd8d017ce84          🙏👍       5  2025-11-29   
1  a6cbfa34-f2b1-4a16-96b6-c94f58cea76f   Very Good       5  2025-11-28   
2  fc67d12c-92e2-45aa-a9e0-011f58a583bc        goof       5  2025-11-28   
3  11306fb9-5571-4950-8d32-604c5402242f       good!       5  2025-11-28   
4  809c46d2-730e-446a-9061-2a45e978ad9d    good jop       5  2025-11-27   

   review_year  review_month bank_code bank_name            user_name  \
0         2025            11       BOA       BOA          Yasin Alemu   
1         2025            11       BOA       BOA          Wariyo Dida   
2         2025            11       BOA       BOA  Hailegebrail Tegegn   
3         2025            11       BOA       BOA            Tsegay ab   
4         2025            11       BOA       BOA       Yohanis Fikadu   

   thumbs_up  text_length       source  
0          0            2  Google Play  
1          0

KeyError: "['date'] not in index"